In [1]:
import torch
from torch import nn
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
import torch.nn.utils.prune as prune

import torchvision
from torchvision import datasets
from torchvision import transforms
from torchmetrics import Accuracy

import torch.optim as optim

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import random
import copy
import gc
import math

import warnings
warnings.filterwarnings('ignore')

from pathlib import Path
from resnet_18 import *

In [6]:
DATASET_PATH = '../datasets/imagenette2/train'
MODEL_PATH = 'saves/resnet/imagenette/0_model_lt.pth.tar'
NUM_IMAGES_PER_CLASS = 500

CLASS_NAME_MAPPING = {
    'springer': 'n02102040',
    'church': 'n03028079',
    'parachute': 'n03888257'
}
TARGET_CLASSES_FRIENDLY_NAMES = list(CLASS_NAME_MAPPING.keys())
TARGET_IMAGES_BY_FOLDER = {folder_name: [] for folder_name in CLASS_NAME_MAPPING.values()}


In [7]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [8]:
model = torch.load(MODEL_PATH, map_location=device, weights_only=False)
model.to(device)
model.eval()

ResNet_features(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (global_pool): AvgPool2d(kernel_size=7, stride=7, padding=0)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, af

In [10]:
preprocess = transforms.Compose([
    transforms.RandomResizedCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],std=[0.229, 0.224, 0.225])])

dataset = datasets.ImageFolder(DATASET_PATH, transform = preprocess)
dataloader = DataLoader(dataset, batch_size = 32, shuffle=False, num_workers=4, pin_memory=True)

In [11]:
# Get the mapping from the actual folder names to the class index
class_name_to_idx = dataset.class_to_idx
# Create a reverse mapping from index to folder name for lookups
idx_to_class_name = {v: k for k, v in class_name_to_idx.items()}

In [12]:
# Get the integer indices for our target classes
target_class_indices = {class_name_to_idx[folder_name] for folder_name in CLASS_NAME_MAPPING.values()}

In [13]:
from tqdm import tqdm
print("Starting image collection...")
pbar = tqdm(dataloader, unit="batch")

def all_targets_met():
    return all(len(images) >= NUM_IMAGES_PER_CLASS for images in TARGET_IMAGES_BY_FOLDER.values())

with torch.no_grad():
    for images, labels in pbar:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, predicted_indices = torch.max(outputs, 1)
        correct_predictions = (predicted_indices == labels)

        for i in range(len(correct_predictions)):
            if correct_predictions[i]:
                true_label_idx = labels[i].item()
                if true_label_idx in target_class_indices:
                    folder_name = idx_to_class_name[true_label_idx]
                    if len(TARGET_IMAGES_BY_FOLDER[folder_name]) < NUM_IMAGES_PER_CLASS:
                        image_np = images[i].cpu().numpy()
                        TARGET_IMAGES_BY_FOLDER[folder_name].append(image_np)

                        # Update progress description with friendly names
                        progress_desc = "Collected: "
                        for friendly_name, f_name in CLASS_NAME_MAPPING.items():
                            collected_count = len(TARGET_IMAGES_BY_FOLDER[f_name])
                            progress_desc += f"{friendly_name}: {collected_count}/{NUM_IMAGES_PER_CLASS} | "
                        pbar.set_description(progress_desc.strip(" | "))

        if all_targets_met():
            print("\nAll target image counts have been met.")
            break

print("\n--- Collection Summary ---")
for friendly_name, folder_name in CLASS_NAME_MAPPING.items():
    collected_count = len(TARGET_IMAGES_BY_FOLDER[folder_name])
    print(f"Collected {collected_count} images for class '{friendly_name}'.")

Starting image collection...


Collected: springer: 500/500 | church: 500/500 | parachute: 500/500:  96%|███████▋| 283/296 [00:42<00:01,  9.20batch/s]


All target image counts have been met.


Collected: springer: 500/500 | church: 500/500 | parachute: 500/500:  96%|███████▋| 283/296 [00:43<00:02,  6.49batch/s]


--- Collection Summary ---
Collected 500 images for class 'springer'.
Collected 500 images for class 'church'.
Collected 500 images for class 'parachute'.


In [16]:
import os
print("\nSaving collected images to .npz files...")
output_dir = 'collected_images'
os.makedirs(output_dir, exist_ok=True)

for friendly_name, folder_name in CLASS_NAME_MAPPING.items():
    image_list = TARGET_IMAGES_BY_FOLDER[folder_name]
    if not image_list:
        print(f"Warning: No images were collected for '{friendly_name}'. Skipping file save.")
        continue

    image_array = np.stack(image_list, axis=0)
    file_path = os.path.join(output_dir, f'{friendly_name}.npz')
    np.savez_compressed(file_path, images=image_array)
    
    print(f"Successfully saved {image_array.shape[0]} images to '{file_path}'")
    print(f" -> Array shape: {image_array.shape}")
    print(f" -> Data type: {image_array.dtype}")

print("\nProcess finished.")


Saving collected images to .npz files...
Successfully saved 500 images to 'collected_images\springer.npz'
 -> Array shape: (500, 3, 224, 224)
 -> Data type: float32
Successfully saved 500 images to 'collected_images\church.npz'
 -> Array shape: (500, 3, 224, 224)
 -> Data type: float32
Successfully saved 500 images to 'collected_images\parachute.npz'
 -> Array shape: (500, 3, 224, 224)
 -> Data type: float32

Process finished.
